### QuSciTech-Labs — Navigation

[Public Labs](https://github.com/jopaneur/quscitech-labs) ·
[Full Edition Access](https://github.com/jopaneur/quscitech-labs#-full-edition-kdp) ·
[Private Repo](https://github.com/jopaneur/quscitech-labs-full) ·
[QuSciTech.com](https://www.quscitech.com) ·
[The Quantum AI Book (QAIS)](https://www.amazon.com/dp/placeholder) ·

DOI: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.17212825.svg)](https://doi.org/10.5281/zenodo.17212825)

### E.2 Lab 2 — ZZ Expectation Scan (QAOA-style Observable)

### Lab Access and Execution Guide
This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional Volume).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.  

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_2_Operations_&_Scientific_Framework_of_QAIS_Advanced_Challenge_Bloch_Trajectories_Under_Composite_Gates.ipynb)



---
**Note for Lab Participants**
Each plot generated in this notebook is automatically saved as a `.png` file under: Advanced_Labs/figures/

The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  

This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**, the images are created inside the session’s working directory at:  
  `/content/Advanced_Labs/figures/`  

- When running **locally**, they appear next to your notebook files, under the subfolder:  
  `Advanced_Labs/figures/`  

- These images are **not automatically added to your GitHub repo**. They will only appear there if you manually copy, commit, and push them.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Book Reference: Chapter 8 — Optimization and Control in Quantum AI Systems**

*Chapter 8*  introduces optimization and control as the operational heart of Quantum AI Systems (QAIS). It explains how variational algorithms—particularly the Quantum Approximate Optimization Algorithm (QAOA) and related parameterized circuits—navigate high-dimensional landscapes by tuning quantum gate parameters to minimize a cost function. These methods blend classical feedback with quantum evolution, demonstrating how iterative learning refines control over interference and entanglement.

A key diagnostic in this framework is the expectation value scan, which measures how observables such as ⟨ZZ⟩ change as a circuit parameter (e.g., rotation angle γ) is varied. Smooth oscillations in expectation values reveal constructive and destructive interference patterns that guide optimization toward energy minima or decision thresholds.

**Beginner Lab 7 — ZZ Expectation Scan (QAOA-style Observable): Oscillating Expectation Values**

Beginner Lab 7 provides a focused demonstration of this process. By sweeping a single control angle through a two-qubit circuit containing ZZ interactions, learners observe how the measured expectation value ⟨ZZ⟩ oscillates periodically, mapping the interference structure that underlies variational optimization. The lab illustrates that control in QAIS is not random adjustment but an intentional tuning of quantum interference to steer learning and reasoning outcomes.

*Goal:* Perform a parameter scan of a two-qubit circuit containing ZZ interactions and measure the observable ⟨ZZ⟩ as a function of the rotation angle. Learners implement a loop over a range of angles, execute the circuit for each setting, and record the resulting expectation values.

**Expected Outcome**

* The ⟨ZZ⟩ values vary smoothly and periodically with the scan angle, displaying the sinusoidal interference pattern characteristic of QAOA-style circuits. The extrema of these oscillations correspond to constructive and destructive interference, visually linking parameter control to quantum optimization behavior.

This exercise reinforces Chapter 8’s principle that optimization in QAIS emerges from interference engineering—by learning how to tune amplitudes and phases, quantum systems can find solutions efficiently while revealing their inner structure through observable patterns.
Cross-reference: Appendix E.1, Figure E.1.7 — Oscillating ⟨ZZ⟩ Expectation Values.

In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Beginner_Labs/figures",
                  fig=None, ax=None):
    """Save the current/explicit figure with a prefixed label and consistent path."""
    os.makedirs(subdir, exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))
    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print("Saved", outpath)


In [ ]:
# === Environment Setup ===
import sys, subprocess, pkgutil
def ensure(pkg):
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p in ['qiskit','qiskit-aer','matplotlib','numpy','scikit-learn']:
    ensure(p)
import qiskit, numpy as np, matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print('Qiskit:', qiskit.__version__)


---


**Lab 7: ZZ Expectation Scan (QAOA-style Observable) — Oscillating Expectation Values**

**Energy Landscapes and Variational Control**

This lab scans how the expectation value of the ZZ operator changes with circuit parameters. The smooth curve shows how varying an angle alters the measured energy. The flow begins with preparing a two-qubit ansatz, applying parameterized gates, then measuring correlations along the ZZ axis. The plot reveals a continuous energy landscape — some parameter settings produce high correlation, while others reduce it. For undergraduates, this matters because it visualizes the principle behind variational algorithms like QAOA: quantum circuits are steered toward optimal parameters by following such landscapes. Instead of discrete outputs, students see the analog “shape” of quantum optimization.

*Book Reference: Chapter 8 — Optimization and Control in Quantum AI Systems*

Readers explore expectation values across varying angles, modeling the iterative scanning used in QAOA. This experiment supports Chapter 8’s discussion of variational optimization and hybrid control strategies.

**Expected Results**

The expectation value ⟨ZZ⟩ should oscillate smoothly as the scan angle changes, revealing predictable quantum interference patterns useful in optimization.

⟨ZZ⟩ varies smoothly between about +1 and −1 as θ is swept.

Near θ ≈ 0 the qubits are predominantly aligned (⟨ZZ⟩ ≈ +1).

Around θ ≈ π/2 anti-alignment dominates (⟨ZZ⟩ ≈ −1).

The periodic pattern reveals tunable correlation strength and sign.


In [ ]:
# Lab 7 — ZZ Expectation Scan

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
import numpy as np
import matplotlib.pyplot as plt

# Observable: ZZ
Z0Z1 = SparsePauliOp.from_list([("ZZ", 1.0)])

# Simple 2-qubit ansatz with a tunable parameter θ
def qaoa_like(theta: float) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.ry(theta, 0)
    qc.cx(0, 1)
    qc.ry(theta/2, 1)
    return qc

# Sweep θ and compute ⟨ZZ⟩ exactly via statevector
angles = np.linspace(0, np.pi, 11)
vals = []
for th in angles:
    sv = Statevector.from_instruction(qaoa_like(th)).data
    exp = float((sv.conj() @ Z0Z1.to_matrix() @ sv).real)  # real expectation
    vals.append(exp)

# Optional debug print
print(list(zip(np.round(angles, 2), np.round(vals, 3))))

# Plot
plt.plot(angles, vals, marker="o")
plt.xlabel("θ (radians)")
plt.ylabel("⟨ZZ⟩")
plt.title("ZZ Expectation Value vs θ")

# Save with correct IEEE label
fig, ax = plt.gcf(), plt.gca()
save_e_figure("Figure E.1.7", "P1_Lab07_ZZ_Expectation_Scan.png",
              subdir="Beginner_Labs/figures", fig=fig, ax=ax)

plt.show()


Figure E.1.7 — ZZ Expectation Value vs θ.
The curve shows how the expectation value ⟨ZZ⟩ oscillates as the variational angle θ is tuned in a two-qubit ansatz. Peaks near +1 indicate strong correlation, valleys near −1 indicate strong anti-correlation, and intermediate values reflect partial correlations set by the circuit parameters.

**Methodology Analysis**

Prepare a two-qubit circuit with RY rotations and a CNOT entangler, controlled by a single parameter θ. For θ ∈ [0, π], generate the state with a statevector simulator and compute the exact expectation value ⟨ZZ⟩ = ⟨ψ(θ)∣Z ⊗ Z∣ψ(θ)⟩. Plot ⟨ZZ⟩ versus θ to visualize how the circuit’s parameter steers qubit–qubit correlations.

**Technical Analysis (for the Visual)**

The ZZ operator measures whether the qubits yield the same outcome (contributing +1) or opposite outcomes (contributing −1) in the computational basis. The ansatz’s entangling structure creates θ-dependent interference among amplitudes, rotating population between same-parity and opposite-parity subspaces. As θ changes, the state’s Bloch-tensor components along Z ⊗ Z shift, producing the observed sinusoid-like oscillation in ⟨ZZ⟩. This scan is a standard diagnostic in variational workflows (e.g., VQE/QAOA) to confirm a circuit can generate and modulate the required two-qubit correlations.

**Intuition Sidebar**

Picture two coins linked by a hidden gear. At one setting they tend to match, at another they tend to oppose. Turning θ is like turning the gear: you smoothly dial from “same” to “opposite,” with every in-between possibility. The plot is the gauge that shows how strong the linkage is for each setting.

**Conclusion — Lab 7**
This lab demonstrates that a minimal variational circuit can continuously tune two-qubit correlations, observable through ⟨ZZ⟩. The oscillatory response verifies that the ansatz provides a controllable knob for entanglement and correlation strength, a prerequisite for effective parameter training in quantum optimization and learning tasks.

**Key Takeaways**

⟨ZZ⟩ is a compact correlation meter for two qubits.

A single parameter θ can steer correlation sign and strength from +1 to −1.

This tunability is essential for variational quantum algorithms that rely on shaping multi-qubit correlations.

**Congratulations — Lab 7**

Great job finishing Lab 7 — ZZ Expectation Scan! You explored how entanglement strength can be tuned continuously using circuit parameters, and how ⟨ZZ⟩ serves as a diagnostic of correlation between qubits. This practical understanding directly supports the design of variational algorithms and optimization tasks in quantum AI, giving you tools that professionals use to characterize and shape multi-qubit systems.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 8 — Optimization and Control in QAIS**:  
- Questions 1–2 (expectation values and periodicity).  
They reinforce the interference patterns observed in **E.2 Lab 2**.

---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.

---
